# Previsão do saldo de emprego em tecnologia

**Pergunta:** o saldo mensal de emprego formal em tecnologia é previsível? Se
é, com que margem de erro — e um modelo estatístico bate uma regra ingênua?

Este caderno responde isso sobre 234 meses de CAGED (2007-01 a 2026-06),
recortados em tecnologia por **setor** (CNAE) **ou** **ocupação** (CBO).

---

### O que faz um número de previsão ser honesto

Duas coisas, e as duas aparecem aqui:

1. **Intervalo, não ponto.** Saldo mensal de emprego é ruidoso. Dizer "em
   dezembro de 2026 o saldo será −2.121" está errado; "estará entre −6.336 e
   +2.094, com 80% de confiança" está certo.
2. **Comparação com baseline.** A série tem sazonalidade forte, então a regra
   "igual ao mesmo mês do ano passado" já acerta bastante. Um modelo que não
   supere isso não capturou nada além do calendário — e publicá-lo daria falsa
   autoridade a um número que a média histórica já daria.

A seleção do modelo é por **validação em origem móvel**, não por AIC: treina
até um ponto, prevê doze meses, anda o ponto, repete. É o mais próximo de "como
o modelo teria se saído se estivesse no ar".

### De onde vêm os dados

Da camada **gold** publicada no Hugging Face — [`{'Gianpedro/mercado-ti-gold'}`](https://huggingface.co/datasets/Gianpedro/mercado-ti-gold).
São 3,3 MB, públicos, sem credencial. Este caderno roda em qualquer máquina:
não precisa de MinIO, Docker, nem dos 60 GB de microdados brutos.

In [ ]:
# Dependências. No Colab, pandas/numpy/matplotlib/statsmodels já vêm; falta o duckdb.
%pip install --quiet duckdb

# O código do pipeline é reaproveitado, não reescrito: as mesmas funções de
# validação e previsão que geram os números publicados. Uma segunda
# implementação aqui divergiria da primeira, e ninguém notaria qual está certa.
import sys, os, subprocess
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB and not Path("Projeto-CAGED").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/GianSE/Projeto-CAGED.git"], check=True)

raiz = Path("Projeto-CAGED/tasks_python") if NO_COLAB else Path("../tasks_python")
sys.path.insert(0, str(raiz.resolve()))
print("pipeline em:", raiz.resolve())

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GOLD = "https://huggingface.co/datasets/Gianpedro/mercado-ti-gold/resolve/main"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

serie = con.execute(f"""
    SELECT mes, saldo, admissoes, desligamentos
    FROM read_parquet('{GOLD}/serie_mensal.parquet')
    ORDER BY mes
""").df()
serie["mes"] = pd.to_datetime(serie["mes"])
serie = serie.set_index("mes").asfreq("MS")

y = serie["saldo"].dropna()
print(f"{len(y)} meses, de {y.index.min():%Y-%m} a {y.index.max():%Y-%m}")
serie.tail()

## 1. A série

O saldo é a definição oficial do CAGED: `+1` para admissão, `−1` para
desligamento. Somado no mês, é a **geração líquida de emprego formal**.

In [ ]:
plt.rcParams.update({"figure.figsize": (13, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10})

fig, ax = plt.subplots()
ax.fill_between(y.index, y, 0, where=y >= 0, alpha=0.65, label="saldo positivo")
ax.fill_between(y.index, y, 0, where=y < 0, alpha=0.65, color="firebrick",
                label="saldo negativo")
ax.axhline(0, lw=0.8, color="black")
ax.axvline(pd.Timestamp("2020-01-01"), ls=":", color="dimgray", lw=1.2)
ax.annotate("Novo CAGED\n(muda a definição\nda competência)",
            xy=(pd.Timestamp("2020-01-01"), y.max() * 0.72),
            xytext=(pd.Timestamp("2014-06-01"), y.max() * 0.82),
            fontsize=8, color="dimgray",
            arrowprops=dict(arrowstyle="->", color="dimgray", lw=0.8))
ax.set_title("Saldo mensal de emprego em tecnologia — CAGED")
ax.set_ylabel("vagas (líquido)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

print(y.describe().to_string())

## 2. Estacionariedade: por que dois testes, e não um

ADF e KPSS têm **hipóteses nulas opostas**:

| teste | H₀ | rejeitar significa |
|---|---|---|
| ADF | tem raiz unitária (não estacionária) | é estacionária |
| KPSS | é estacionária | não é estacionária |

Rodar só um leva a conclusão apressada. Só há evidência de estacionariedade
quando **ADF rejeita E KPSS não rejeita** — os dois apontando na mesma direção.
Quando discordam, o honesto é dizer que é inconclusivo, não escolher o que
convém.

In [ ]:
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    adf_p = adfuller(y, autolag="AIC")[1]
    kpss_p = kpss(y, regression="c", nlags="auto")[1]

print(f"ADF   p = {adf_p:.4f}   (H0: tem raiz unitária)")
print(f"KPSS  p = {kpss_p:.4f}   (H0: é estacionária)")

if adf_p < 0.05 and kpss_p > 0.05:
    veredito = "ESTACIONÁRIA — os dois testes concordam"
elif adf_p > 0.05 and kpss_p < 0.05:
    veredito = "NÃO ESTACIONÁRIA — os dois testes concordam"
else:
    veredito = "INCONCLUSIVO — os testes discordam"
print(f"\n-> {veredito}")

## 3. Sazonalidade

É o padrão mais forte da série, e é o que torna o baseline sazonal difícil de
bater. Dezembro é estruturalmente negativo (fim de contrato temporário,
encerramento de ciclo); o segundo semestre concentra contratação.

In [ ]:
meses = ["jan", "fev", "mar", "abr", "mai", "jun",
         "jul", "ago", "set", "out", "nov", "dez"]
sazonal = y.groupby(y.index.month).agg(["mean", "std", "count"])
sazonal.index = meses

fig, ax = plt.subplots(figsize=(11, 3.6))
cores = ["steelblue" if v >= 0 else "firebrick" for v in sazonal["mean"]]
ax.bar(sazonal.index, sazonal["mean"], color=cores, alpha=0.85)
ax.errorbar(sazonal.index, sazonal["mean"], yerr=sazonal["std"], fmt="none",
            ecolor="dimgray", elinewidth=0.9, capsize=3)
ax.axhline(0, lw=0.8, color="black")
ax.set_title("Saldo médio por mês do ano (barra de erro = desvio-padrão)")
ax.set_ylabel("vagas")
plt.tight_layout(); plt.show()

print(sazonal.round(0).to_string())

## 4. A emenda de 2020 — uma limitação que precisa ser declarada

A série cruza as **duas gerações do CAGED**, e elas datam a movimentação de
formas diferentes:

- **CAGED antigo (2007–2019)** → competência **declarada**
- **Novo CAGED (2020–2026)** → competência da **movimentação**

Não é a mesma definição. Declaração atrasada aparece no mês da declaração na
primeira e no mês do fato na segunda.

Parte da mudança de nível abaixo é **real** (o ciclo de contratação de
2021–22), parte pode ser convenção de datação. O dado não permite separar as
duas, e por isso a limitação é reportada em vez de escondida — qualquer modelo
sobre esta série carrega essa ressalva.

In [ ]:
corte = pd.Timestamp("2020-01-01")
antes, depois = y[y.index < corte], y[y.index >= corte]

comparacao = pd.DataFrame({
    "período": ["2007–2019 (competência declarada)",
                "2020–2026 (competência do fato)"],
    "meses": [len(antes), len(depois)],
    "média": [antes.mean(), depois.mean()],
    "desvio": [antes.std(), depois.std()],
    "mínimo": [antes.min(), depois.min()],
    "máximo": [antes.max(), depois.max()],
})
print(comparacao.round(0).to_string(index=False))
print(f"\nA média mais que dobra e o desvio cresce {depois.std() / antes.std():.0%}.")

## 5. O baseline obrigatório

A regra: **o saldo deste mês será igual ao do mesmo mês do ano passado.**

Não há modelo nisso — é o calendário. Justamente por isso ela é o piso: um
SARIMA que não a supere está redescobrindo a sazonalidade e chamando de
previsão.

In [ ]:
from ciencia_dados import previsao_saldo as ps

print("candidatos avaliados:")
for ordem, sazonal_ in ps.CANDIDATOS:
    print(f"   SARIMA{ordem}{sazonal_}")
print("\nA grade é pequena de propósito: com 234 pontos e várias dobras, uma")
print("grade grande acharia por acaso o modelo que se sai bem no teste — que é")
print("o mesmo overfitting, só que na validação.")

## 6. Validação em origem móvel

Treina até um ponto, prevê 12 meses, compara com o observado, anda o ponto 12
meses para frente e repete — cinco dobras. O baseline entra nas **mesmas
dobras**; comparar com um baseline avaliado de outro jeito seria comparar com
nada.

In [ ]:
placar = ps.validar(y, h=12, origens=5)
print(placar.to_string(index=False))

vencedor = placar.iloc[0]
if vencedor["modelo"] == "naive sazonal":
    print("\n⚠️  Nenhum SARIMA superou a regra ingênua. A leitura honesta seria")
    print("    publicar a média sazonal e dizer que o resto não acrescenta.")
    ordem, sazonal_ = ps.CANDIDATOS[0]
else:
    i = [f"SARIMA{o}{s}" for o, s in ps.CANDIDATOS].index(vencedor["modelo"])
    ordem, sazonal_ = ps.CANDIDATOS[i]
    print(f"\nEscolhido: {vencedor['modelo']}")
    print(f"MAE {vencedor['mae']:,.0f} contra {placar.loc[placar.modelo == 'naive sazonal', 'mae'].iloc[0]:,.0f} do baseline")
    print(f"-> {vencedor['ganho_vs_naive_%']:+.1f}% de erro a menos")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ordenado = placar.sort_values("mae")
cores = ["seagreen" if m != "naive sazonal" else "dimgray"
         for m in ordenado["modelo"]]
ax.barh(ordenado["modelo"], ordenado["mae"], color=cores, alpha=0.85)
base = placar.loc[placar.modelo == "naive sazonal", "mae"].iloc[0]
ax.axvline(base, ls="--", color="black", lw=1.2, label="baseline (naive sazonal)")
ax.set_xlabel("MAE fora da amostra (vagas)")
ax.set_title("Erro de cada candidato — barra à esquerda da linha vence o baseline")
ax.legend(fontsize=8); ax.invert_yaxis()
plt.tight_layout(); plt.show()

Repare que **um dos candidatos fica à direita da linha** — ele é *pior* que a
regra ingênua. Sem o baseline na tabela, seria possível publicá-lo como
"modelo SARIMA ajustado" e ninguém perceberia que a média histórica acertaria
mais.

## 7. Diagnóstico dos resíduos

Aqui está o que um dashboard não mostra e uma dissertação precisa.

Se o modelo capturou a estrutura da série, o que sobra deve ser **ruído**: sem
autocorrelação, centrado em zero, variância estável. Resíduo com padrão
significa que há informação na série que o modelo deixou na mesa — e o
intervalo de confiança reportado está **otimista demais**.

O teste de Ljung-Box formaliza: H₀ é "os resíduos não têm autocorrelação". Aqui
**não rejeitar é o bom resultado** — o inverso do que costuma valer.

In [ ]:
ajuste = ps._ajustar(y, ordem, sazonal_)
residuo = pd.Series(ajuste.resid, index=y.index).iloc[13:]  # descarta o aquecimento

from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(residuo, lags=[6, 12, 24], return_df=True)
print(lb.round(4).to_string())
print()
for lag, p in zip(lb.index, lb["lb_pvalue"]):
    situacao = "ok (sem autocorrelação)" if p > 0.05 else "ATENÇÃO: há estrutura sobrando"
    print(f"   lag {lag:>2}: p = {p:.4f}  ->  {situacao}")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, eixos = plt.subplots(2, 2, figsize=(13, 7))

eixos[0, 0].plot(residuo.index, residuo, lw=0.9)
eixos[0, 0].axhline(0, color="black", lw=0.8)
eixos[0, 0].set_title("Resíduo ao longo do tempo")

plot_acf(residuo, lags=36, ax=eixos[0, 1], zero=False)
eixos[0, 1].set_title("Autocorrelação do resíduo (faixa azul = ruído)")

eixos[1, 0].hist(residuo, bins=28, alpha=0.8, color="steelblue")
eixos[1, 0].axvline(residuo.mean(), color="firebrick", ls="--",
                    label=f"média {residuo.mean():,.0f}")
eixos[1, 0].set_title("Distribuição do resíduo")
eixos[1, 0].legend(fontsize=8)

from scipy import stats
stats.probplot(residuo, dist="norm", plot=eixos[1, 1])
eixos[1, 1].set_title("Q-Q normal — desvio nas pontas = cauda gorda")

plt.tight_layout(); plt.show()

print(f"média do resíduo: {residuo.mean():,.1f}  (deve ser ~0)")
print(f"desvio:           {residuo.std():,.1f}")
print(f"assimetria:       {stats.skew(residuo):.2f}   curtose: {stats.kurtosis(residuo):.2f}")

**Como ler o Q-Q:** pontos sobre a reta significam resíduo normal. Desvio nas
extremidades indica **cauda mais gorda que a normal** — choques extremos
(pandemia, crise) mais frequentes do que a distribuição normal prevê.

Consequência prática, e ela importa: o intervalo de confiança do SARIMA assume
normalidade. Com cauda gorda, o intervalo real é **mais largo** que o
reportado. Ou seja, a incerteza mostrada adiante é o **piso** dela, não o teto.

## 8. A previsão

Com intervalo de 80%, que é o usual em análise de conjuntura.

In [ ]:
previsao, _ = ps.prever(y, ordem, sazonal_, h=18, alpha=0.20)
previsao.index.name = "mes"
print(previsao.round(0).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.6))
hist = y.tail(60)
ax.plot(hist.index, hist, lw=1.6, label="observado")
ax.fill_between(previsao.index, previsao["inferior"], previsao["superior"],
                alpha=0.22, color="darkorange", label="intervalo de 80%")
ax.plot(previsao.index, previsao["previsao"], lw=2, ls="--", color="darkorange",
        marker="o", ms=3.5, label="previsto")
ax.axhline(0, lw=0.8, color="black")
ax.set_title(f"Previsão do saldo mensal — {vencedor['modelo']}")
ax.set_ylabel("vagas (líquido)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

total = previsao["previsao"].sum()
print(f"acumulado em {len(previsao)} meses: {total:,.0f} vagas")
print(f"faixa de 80%: {previsao['inferior'].sum():,.0f} a {previsao['superior'].sum():,.0f}")
print()
print("A faixa acumulada é enorme porque os erros mensais se somam. É por isso")
print("que previsão de emprego se lê em direção e ordem de grandeza, nunca no")
print("número exato.")

## 9. Limitações

Quatro, em ordem de gravidade:

1. **A emenda de 2020.** A série cruza duas convenções de datação da
   competência. O modelo trata a mudança de nível como parte do processo, não
   como quebra declarada. Alternativas: variável de nível, ou modelar só
   2020+ — perde 13 anos de histórico, ganha homogeneidade.

2. **Cauda gorda no resíduo.** O intervalo do SARIMA supõe normalidade; com
   choques extremos mais frequentes que o normal, a incerteza real é maior que
   a reportada.

3. **Seis meses de 2026 podem ser revisados.** O CAGED revisa competências
   recentes conforme declarações atrasadas entram. Os últimos meses da série
   são preliminares.

4. **Previsão univariada.** O modelo olha só o próprio passado da série. Não
   usa juros, PIB, câmbio nem nada externo — então não antecipa choque que
   venha de fora. Extensão natural: SARIMAX com variável exógena.

---

### Reprodutibilidade

Este caderno lê a camada gold publicada e importa as mesmas funções
(`ciencia_dados.previsao_saldo`) que produzem os números do dashboard. Não há
uma segunda implementação: se o pipeline mudar, este caderno muda com ele.

Para conferir qualquer número daqui, basta reexecutar — o dado é público.